# Zadanie 3
GG - W dużej mierze AI pomagało mi głównie łapać błędy, literówki, naprawiać błędy składni pythona i refaktoryzować niektóre funkcje (zaznaczone gdzie). 

In [ ]:
#wycena opcji, kawałek pliku master
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from lib.dates import year_fraction
from lib.import_data import load_market, MarketEngine
from lib.vanilla_option import OptionType, VanillaOptionBlackSholes, d,strike_for_delta, vanna_volga_cost_coefficients
from lib.binary_option import CoNBlackScholes, AoNBlackScholes

class PricingEngine:
    def __init__(self, market_file: str, tenor: str, imply_pln: bool):
        self.market = load_market(market_file, tenor)
        self.m_engine = MarketEngine(self.market)

        self.r_pln, self.r_eur, self.df_d, self.df_f = (self.m_engine.rates_dfs(imply_pln))

        self.sigma_atm = self.market.atm
        self.sigma_25C = self.market.atm + self.market.bf25 + 0.5 * self.market.rr25
        self.sigma_25P = self.market.atm + self.market.bf25 - 0.5 * self.market.rr25
        
    def strike(self,delta: float,option_type: OptionType):
        sigma = self.sigma_atm
        return strike_for_delta(
            delta=delta,
            option_type=option_type,
            df_d=self.df_d,
            df_f=self.df_f,
            S_t=self.market.spot,
            sigma=sigma,
            t=self.market.start,
            T=self.market.expiry,
            base=365,
            forward=self.market.delta_forward,
            premium=self.market.delta_premium,
        )
         
    def vanilla_price(self,K: float,option_type: OptionType,pricing_method: str,p: float = 1.0,):
        option = VanillaOptionBlackSholes(T=self.market.expiry,K=K,option_type=option_type)
        
        if pricing_method.lower() == "bs":
            return option.price(
                df_d=self.df_d,
                df_f=self.df_f,
                S_t=self.market.spot,
                sigma=self.sigma_atm,
                t=self.market.start,
                base=365,
            )

        elif pricing_method.lower() == "vv":
            return option.vanna_volga_price(
                df_d=self.df_d,
                df_f=self.df_f,
                p=p,
                S_t=self.market.spot,
                sigma_atm=self.sigma_atm,
                sigma_25C=self.sigma_25C,
                sigma_25P=self.sigma_25P,
                t=self.market.start,
                delta_forward=self.market.delta_forward,
                delta_premium=self.market.delta_premium,
                base=365,
            )

        raise ValueError("Metoda to nie BS ani VV")

    def con_price(self,K: float,option_type: OptionType,pricing_method: str = "bs",p: float = 1.0,dK: float = 0.0001,):
        option = CoNBlackScholes(T=self.market.expiry, K=K, option_type=option_type )
        if pricing_method.lower() == "bs":
            return option.price(
                df_d=self.df_d,
                df_f=self.df_f,
                S_t=self.market.spot,
                sigma=self.sigma_atm,
                t=self.market.start,
                base=365,
            )
        elif pricing_method.lower() == "vv":
            return option.vanna_volga_price(
                df_d=self.df_d,
                df_f=self.df_f,
                p=p,
                S_t=self.market.spot,
                sigma_atm=self.sigma_atm,
                sigma_25C=self.sigma_25C,
                sigma_25P=self.sigma_25P,
                t=self.market.start,
                delta_forward=self.market.delta_forward,
                delta_premium=self.market.delta_premium,
                base=365,
                dK=dK,
            )

        raise ValueError("Metoda to nie BS ani VV")

    def aon_price(self,K: float,option_type: OptionType,pricing_method: str = "bs",p: float = 1.0,dK: float = 0.0001):
        option = AoNBlackScholes(T=self.market.expiry,K=K,option_type=option_type)
        
        if pricing_method.lower() == "bs":
            return option.price(
                df_d=self.df_d,
                df_f=self.df_f,
                S_t=self.market.spot,
                sigma=self.sigma_atm,
                t=self.market.start,
                base=365,
            )

        elif pricing_method.lower() == "vv":
            return option.vanna_volga_price(
                df_d=self.df_d,
                df_f=self.df_f,
                p=p,
                S_t=self.market.spot,
                sigma_atm=self.sigma_atm,
                sigma_25C=self.sigma_25C,
                sigma_25P=self.sigma_25P,
                t=self.market.start,
                delta_forward=self.market.delta_forward,
                delta_premium=self.market.delta_premium,
                base=365,
                dK=dK,
            )

        raise ValueError("Metoda to nie BS ani VV")

## Turbo Depozyt
Rozważamy lokatę o następującej wypłacie:
$$
N \left( 1 + \delta(T)\left(R_{\min} \mathbf{1}_{\{S_T < L\}} + R_{\mathrm{mkt}} \mathbf{1}_{\{L < S_T < U\}} + R_{\max} \mathbf{1}_{\{U < S_T\}} \right) \right)
$$

Dla punktu (a), liczymy dla waluty niebazowej (PLN), więc używamy CoN oraz implikujemy EUR. 

Dla punktu (b), liczymy dla walucie bazowej (EUR), więc używamy AoN oraz implikujemy PLN.

Wystawiający bierze nominał $N$ oraz marżę $m*N$, a w zamian daje produkt, którego wartość to:
wartość bieżąca rynku + wartość bieżąca wbudowanych opcji:

$$
N(1+m)
=
PV\Big(
N(1+\delta(T)R_{\mathrm{mkt}}) - N\delta(T)(R_{\mathrm{mkt}} - R_{\min})\mathbf{1}_{\{S_T < L\}} + N\delta(T)(R_{\max} -
R_{\mathrm{mkt}})\mathbf{1}_{\{S_T > U\}} \Big)
$$

Rozdzielając składniki:

$$
N + mN = PV\big(N(1+\delta(T)R_{\mathrm{mkt}})\big) + PV\Big( - N\delta(T)(R_{\mathrm{mkt}} - R_{\min})\mathbf{1}_{\{S_T < L\}}
+ N\delta(T)(R_{\max} - R_{\mathrm{mkt}})\mathbf{1}_{\{S_T > U\}} \Big)
$$

Ponieważ:

$$
N = PV\big(N(1+\phi(T)R_{\mathrm{mkt}})\big)
$$

oraz:

$$
PV(\mathbf{1}_{\{S_T < L\}}) = \text{Put}_{\mathrm{binary}}(L), 
PV(\mathbf{1}_{\{S_T > U\}}) = \text{Call}_{\mathrm{binary}}(U)
$$

to:

$$
mN = - N\phi(T)(R_{\mathrm{mkt}} - R_{\min}) \text{Put}_{\mathrm{binary}}(L) + N\phi(T)(R_{\max} - R_{\mathrm{mkt}}) \text{Call}_{\mathrm{binary}}(U)
$$

Dzieląc przez $N$:

$$
m = \phi(T)\Big( -(R_{\mathrm{mkt}} - R_{\min}) \text{Put}_{\mathrm{binary}}(L) + (R_{\max} - R_{\mathrm{mkt}}) \text{Call}_{\mathrm{binary}}(U) \Big)
$$

Stąd:

$$
\text{Put}_{\mathrm{binary}}(L)
=
\frac{
(R_{\max} - R_{\mathrm{mkt}})\text{Call}_{\mathrm{binary}}(U) - \frac{m}{\delta(T)}
}{
(R_{\mathrm{mkt}} - R_{\min})
}
$$

### Wycena opcji wbudowanych
W zależności od tego czy liczymy w walucie bazowej czy niebazowej funkcja znajduje opcje CoN lub AoN dla odpowiednich strików. 
Do wyceny opcji korzystam z zadania 1, biorąc wycenę dilerską metodę VV.

In [ ]:
def turbo_depozyt_opcje(wersja_a:bool,tenor:str, L,U):
    if wersja_a:
        engine = PricingEngine("market_data.xlsx", tenor=tenor,imply_pln=False)  
        put_L = engine.con_price(K=L, option_type=OptionType.PUT, pricing_method="vv" )
        call_U = engine.con_price(K=U, option_type=OptionType.CALL, pricing_method="vv" )
    else:
        engine = PricingEngine("market_data.xlsx", tenor=tenor,imply_pln=True)  
        put_L=engine.aon_price(K=L, option_type=OptionType.PUT, pricing_method="vv" )
        call_U=engine.aon_price(K=U, option_type=OptionType.CALL, pricing_method="vv" )   
    return(put_L,call_U)

put_L,call_U=turbo_depozyt_opcje(wersja_a=True,tenor="3M",L=4.13,U=4.4)
print (f"Wartość opcji binarnych: put_L {put_L:.6f}, call_U {call_U:.6f}")

### Znajdowanie L oraz U
Posiadając parametry dotyczące tenoru, R_min, R_max, R_mkt, marży m oraz marży U, funkcja znajduje L oraz U.
Najpierw wczytywany jest rynek, znajdowane U oraz z użyciem poprzedniej funkcji znajdowana cena opcji binarnej ze strikiem U.
Następnie z użyciem formuły wyprowadzonej wyżej znajdowana jest cena binarnego puta od L.
Potem znajdowany jest L poprzez szukanie wartości dla wielu innych L_i w przedziale (3.5, 5.5). Dla najbliższego L~L_i, uznajemny, że cena dla L jest właśnie tą ceną L_i. 

In [ ]:
def turbo_depozyt_znajdz_L_U(wersja_a:bool,T:str,R_min,R_max,R_mkt,m,U_margin):
    if wersja_a:
        engine = PricingEngine("market_data.xlsx", tenor=T,imply_pln=False) 
    else: 
        engine = PricingEngine("market_data.xlsx", tenor=T,imply_pln=True) 
      
    tau_vol, tau_pln, tau_eur = engine.m_engine.taus() 
    U = U_margin+engine.market.forward
    x,call_U = turbo_depozyt_opcje(wersja_a,T,4,U) #x oraz 4 , bo nie znamy opcji, a nie chcemy aby wyskoczył błąd
    if wersja_a:
        Put_binary_price=((R_max-R_mkt)*call_U-m/tau_eur)/(R_mkt-R_min)
    else:                  
        Put_binary_price=((R_max-R_mkt)*call_U-m/tau_pln)/(R_mkt-R_min)
        
    Ls = np.linspace(3.5, 5.5, 1000)
    ceny = []
    for L in Ls:
        if wersja_a:
            l = engine.con_price( K=L, option_type=OptionType.PUT, pricing_method="vv")
        else:
            l = engine.aon_price( K=L, option_type=OptionType.PUT, pricing_method="vv")
        ceny.append(abs(l - Put_binary_price))
    indeks = np.argmin(ceny)
    L = float(Ls[indeks])
    return(L,U)

L, U=turbo_depozyt_znajdz_L_U(wersja_a=True,T="3M",R_min=0.04,R_max=0.08,R_mkt=0.059,m=5/10000,U_margin=0.125)
print(f"Wyznaczona wartość L to: {L:.6f}")

### Wykres wartości lokaty
Dla lokaty ze znanymi parametrami na przedziale (L-0.2, U+0.2), wyliczane są wypłaty opcji i depozytu, bezpośrednio z definicji.
Wyliczany jest również nadwyżkowy zysk w porówaniu z lokatą rynkową. Następnie oba wyliczenia są przedstawiane na wykresie.

In [ ]:
def wartosc_lokaty(wersja_a:bool,T:str,R_min,R_max,R_mkt,m,L,U):
    if wersja_a:
        engine = PricingEngine("market_data.xlsx", tenor=T,imply_pln=wersja_a) 
    else: 
        engine = PricingEngine("market_data.xlsx", tenor=T,imply_pln=False)   
    tau_vol, tau_pln, tau_eur = engine.m_engine.taus()
    if wersja_a: tau=tau_eur
    else:tau=tau_pln
        
    S_Ts = np.linspace(L-0.2, U+0.2, 2000)
    wartosc = []
    zysk=[]
    
    for S_T in S_Ts:
        if S_T<=L:
            wyplata=1+R_min*tau
            z=wyplata-(1+R_mkt*tau)*(1+m)
        elif S_T<U and S_T>L: 
            wyplata=1+R_mkt*tau
            z=wyplata-(1+R_mkt*tau)*(1+m)
        else: 
            wyplata=1+tau*R_max
            z=wyplata-(1+R_mkt*tau)*(1+m)
        wartosc.append(wyplata)
        zysk.append(z)
    plt.figure()
    plt.plot(S_Ts, wartosc, label="Wartość lokaty")
    plt.axhline((1+R_mkt*tau)*(1+m),linestyle="--", color="grey", label="Lokata rynkowa")
    plt.axvline(L, linestyle="--", color="red", label="L")
    plt.axvline(U, linestyle="--", color="green", label="U")
    plt.xlabel("Końcowy kurs")
    plt.ylabel("Wartość depozytu")
        
    plt.title("Wartość turbo depozytu w chwili T")
    plt.legend()
    plt.show()
    
    plt.figure()
    plt.gca().yaxis.set_major_formatter(PercentFormatter(1))
    plt.plot(S_Ts, zysk, label="Nadwyżkowy zysk z lokaty")
    plt.axhline(0,linestyle="--", color="grey", label="Lokata rynkowa")
    plt.axvline(L, linestyle="--", color="red", label="L")
    plt.axvline(U, linestyle="--", color="green", label="U")
    plt.xlabel("Końcowy kurs")
    plt.ylabel("Nadwyżkowy zysk")
        
    plt.title("Nadwyżkowy zysk względem depozytu w chwili T")
    plt.legend()
    plt.show()
    
wartosc_lokaty(wersja_a=True,T="3M",R_min=0.04,R_max=0.08,R_mkt=0.059,m=5/10000,L=L,U=U)